# 03 - XGBoost Modeling

This notebook trains an XGBoost classifier on the same processed dataset and stratified holdout split used by the logistic-regression baseline. Its purpose is to model nonlinearities and feature interactions that a linear baseline may miss.

In [1]:
import json
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import plotly.graph_objects as go
from sklearn.metrics import average_precision_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold, train_test_split
from xgboost import XGBClassifier

from churn_ml.models.evaluate_model import classification_metrics

RANDOM_STATE = 42
TARGET_COLUMN = "Churn Value"
CHURN_THRESHOLD = None  # Set a value from 0 to 1 to override the training churn-rate threshold.
# CHURN_THRESHOLD = 0.5  # Set a value from 0 to 1 to override the training churn-rate threshold.
def find_project_file(relative_path):
    for directory in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        candidate = directory / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find {relative_path} from {Path.cwd()} or its parent directories.")

DATA_PATH = find_project_file(Path("data/processed/prediction_df_xgboost.csv"))
VALUE_DATA_PATH = find_project_file(Path("data/raw/Telco_customer_churn.csv"))

model_df = pd.read_csv(DATA_PATH)
value_df = pd.read_csv(VALUE_DATA_PATH, usecols=["CustomerID", TARGET_COLUMN, "CLTV"])
assert model_df.columns[-1] == TARGET_COLUMN, "The target must be the final column."
X = model_df.drop(columns=TARGET_COLUMN)
y = model_df[TARGET_COLUMN]
if len(value_df) != len(model_df) or not value_df[TARGET_COLUMN].equals(y):
    raise ValueError("Raw CLTV values are not aligned with the processed modeling data.")
customer_ltv = value_df.set_index("CustomerID")["CLTV"].rename("predicted_ltv_if_retained")
customer_ids = value_df["CustomerID"]
X_train, X_test, y_train, y_test, ltv_train, ltv_test, customer_id_train, customer_id_test = train_test_split(
    X, y, customer_ltv, customer_ids, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
decision_threshold = float(y_train.mean()) if CHURN_THRESHOLD is None else float(CHURN_THRESHOLD)
if not 0 < decision_threshold < 1:
    raise ValueError("CHURN_THRESHOLD must be between 0 and 1.")
print(f"Training rows: {len(X_train):,}; test rows: {len(X_test):,}")
print(f"Positive-class weight: {scale_pos_weight:.2f}")
print(f"Prediction threshold: {decision_threshold:.1%}")

optuna.logging.set_verbosity(optuna.logging.WARNING)

Training rows: 5,634; test rows: 1,409
Positive-class weight: 2.77
Prediction threshold: 26.5%


## Hyperparameter tuning and evaluation

Optuna tunes only on the training partition using shuffled, stratified cross-validation. The test partition remains untouched until the final evaluation. Each trial maximizes cross-validated PR-AUC (average precision), which focuses on ranking the minority churn class. XGBoost uses `eval_metric="aucpr"` to monitor the same quantity during fold-level early stopping. The probability objective remains `objective="binary:logistic"`; `aucpr` is an evaluation metric, not an XGBoost objective.

Each fold uses early stopping, and the final model uses the median number of trees selected across folds. XGBoost natively handles the missing values in `Total Charges`; the missingness indicator remains available as an explicit feature.

In [2]:
N_TRIALS = 50
N_SPLITS = 5
EARLY_STOPPING_ROUNDS = 100
MAX_ESTIMATORS = 2_000

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

def objective(trial):
    params = {
        "objective": "binary:logistic",
        "eval_metric": "aucpr",
        "n_estimators": MAX_ESTIMATORS,
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.20, log=True),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 20.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.60, 1.00),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.60, 1.00),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 20.0, log=True),
        "scale_pos_weight_multiplier": trial.suggest_float("scale_pos_weight_multiplier", 0.5, 1.5),
        "random_state": RANDOM_STATE,
        "n_jobs": 1,
        "tree_method": "hist",
        "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
    }
    weight_multiplier = params.pop("scale_pos_weight_multiplier")
    fold_scores, best_iterations = [], []

    for train_index, valid_index in cv.split(X_train, y_train):
        X_fold_train, X_fold_valid = X_train.iloc[train_index], X_train.iloc[valid_index]
        y_fold_train, y_fold_valid = y_train.iloc[train_index], y_train.iloc[valid_index]
        fold_scale_pos_weight = (y_fold_train == 0).sum() / (y_fold_train == 1).sum()
        model = XGBClassifier(
            **params,
            scale_pos_weight=fold_scale_pos_weight * weight_multiplier,
        )
        model.fit(X_fold_train, y_fold_train, eval_set=[(X_fold_valid, y_fold_valid)], verbose=False)
        fold_scores.append(average_precision_score(y_fold_valid, model.predict_proba(X_fold_valid)[:, 1]))
        best_iterations.append(model.best_iteration + 1)

    trial.set_user_attr("best_iterations", best_iterations)
    return float(np.mean(fold_scores))

sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE, multivariate=True)
study = optuna.create_study(direction="maximize", sampler=sampler, study_name="xgboost_pr_auc")
study.optimize(objective, n_trials=N_TRIALS, gc_after_trial=True, show_progress_bar=True)

best_params = study.best_trial.params.copy()
best_n_estimators = int(np.median(study.best_trial.user_attrs["best_iterations"]))
best_weight_multiplier = best_params.pop("scale_pos_weight_multiplier")
tuned_xgb_config = {
    **best_params,
    "n_estimators": best_n_estimators,
    "scale_pos_weight_multiplier": best_weight_multiplier,
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "cv_pr_auc": study.best_value,
}
OPTUNA_PARAMS_PATH = DATA_PATH.parents[2] / "artifacts/models/xgboost_optuna_best_params.json"
OPTUNA_PARAMS_PATH.parent.mkdir(parents=True, exist_ok=True)
with OPTUNA_PARAMS_PATH.open("w", encoding="utf-8") as file:
    json.dump(tuned_xgb_config, file, indent=2)
print(f"Saved tuned XGBoost configuration to {OPTUNA_PARAMS_PATH}")
display(pd.DataFrame([{**study.best_trial.params, "cv_pr_auc": study.best_value, "n_estimators": best_n_estimators}]))

xgb_model = XGBClassifier(
    **best_params,
    objective="binary:logistic",
    eval_metric="aucpr",
    n_estimators=best_n_estimators,
    scale_pos_weight=scale_pos_weight * best_weight_multiplier,
    random_state=RANDOM_STATE,
    n_jobs=1,
    tree_method="hist",
)
xgb_model.fit(X_train, y_train)

y_proba = xgb_model.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= decision_threshold).astype(int)
xgb_metrics = pd.Series(classification_metrics(y_test, y_pred, y_proba), name="xgboost")
display(xgb_metrics.to_frame())

confusion = confusion_matrix(y_test, y_pred, labels=[0, 1])
fig = go.Figure(go.Heatmap(
    z=confusion, x=["Predicted: no churn", "Predicted: churn"],
    y=["Actual: no churn", "Actual: churn"],
    colorscale="Blues", text=confusion, texttemplate="%{text}",
    colorbar={"title": "Customers"},
))
fig.update_layout(title=f"XGBoost Confusion Matrix (threshold = {decision_threshold:.1%})")
fig.show()

c:\H_Drive\MyPrograms\miniconda\miniconda_win\envs\churn_ml_env001\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(


  0%|          | 0/50 [00:00<?, ?it/s]

Saved tuned XGBoost configuration to W:\Workstation ExtDrive\007 Data Science\001 Data Science Training\2026_016 ML Churn Model End to End\artifacts\models\xgboost_optuna_best_params.json


,max_depth,learning_rate,min_child_weight,subsample,colsample_bytree,gamma,reg_alpha,reg_lambda,scale_pos_weight_multiplier,cv_pr_auc,n_estimators
0,2,0.07728,11.245397,0.757115,0.865606,0.318342,0.002264,10.062249,1.25371,0.695612,179


,xgboost
accuracy,0.645848
precision,0.424970
recall,0.946524
f1,0.586578
pr_auc,0.672541
roc_auc,0.855912


## Feature importance

Gain-based importance summarizes how much each feature improves tree splits. It is a model-specific association measure, not a causal explanation.

In [3]:
feature_importance = (
    pd.DataFrame({"feature": X_train.columns, "importance": xgb_model.feature_importances_})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
feature_importance.head(15)

,feature,importance
0,Contract_Month-to-month,0.326847
1,Online Security_No,0.107009
2,Internet Service_Fiber optic,0.100093
3,Tech Support_No,0.092753
4,Payment Method_Electronic check,0.064611
5,Dependents,0.047193
6,Contract_Two year,0.044713
7,Streaming Movies_Yes,0.024161
8,Tenure Months,0.023268
9,Internet Service_No,0.023044


# Expected Value of Retention Targeting

This evaluation retrieves the raw dataset's `CLTV` value for every holdout-test customer and treats it as that customer's **predicted lifetime value if retained**. `CLTV` is deliberately not a churn-model feature; it is used only after prediction to prioritize outreach.

The expected net value for each customer is calculated as:

$$P(\text{churn}) \times 10\% \times \text{predicted LTV if retained} - \$20 - (40\% \times \$500)$$

Assumptions: outreach costs **$20** for every targeted customer. The offer costs **$500** only when it is accepted; this scenario assumes a **40% offer-acceptance rate among targeted customers**, including customers who would have stayed without outreach. Its expected cost is therefore $200 per target. The 10% retention uplift is a separate scenario assumption: it represents the share of would-be churners saved by the intervention. Neither assumption is estimated by the churn model, so replace them when campaign data becomes available. The top 100 are selected by expected net value—not merely by churn probability—so high-value customers are prioritized.

In [4]:
OUTREACH_COST = 20
OFFER_COST = 500
RETENTION_UPLIFT = 0.10
OFFER_ACCEPTANCE_RATE = 0.40
TARGET_COUNT = 100

targeting_candidates = pd.DataFrame({
    "CustomerID": customer_id_test.to_numpy(),
    "predicted_churn_probability": y_proba,
    "predicted_ltv_if_retained": ltv_test.to_numpy(),
})
targeting_candidates["expected_value_before_cost"] = (
    targeting_candidates["predicted_churn_probability"]
    * RETENTION_UPLIFT
    * targeting_candidates["predicted_ltv_if_retained"]
)
targeting_candidates["expected_offer_cost"] = OFFER_COST * OFFER_ACCEPTANCE_RATE
targeting_candidates["campaign_cost"] = OUTREACH_COST + targeting_candidates["expected_offer_cost"]
targeting_candidates["expected_net_value"] = (
    targeting_candidates["expected_value_before_cost"]
    - targeting_candidates["campaign_cost"]
)

top_100_targets = (
    targeting_candidates
    .sort_values("expected_net_value", ascending=False)
    .head(TARGET_COUNT)
    .reset_index(drop=True)
)

targeting_summary = pd.DataFrame({
    "customers_targeted": [len(top_100_targets)],
    "expected_value_before_cost": [top_100_targets["expected_value_before_cost"].sum()],
    "outreach_cost": [OUTREACH_COST * len(top_100_targets)],
    "expected_offer_cost": [top_100_targets["expected_offer_cost"].sum()],
    "campaign_cost": [top_100_targets["campaign_cost"].sum()],
    "expected_net_value": [top_100_targets["expected_net_value"].sum()],
})
display(targeting_summary.style.format({
    "expected_value_before_cost": "${:,.2f}",
    "outreach_cost": "${:,.2f}",
    "expected_offer_cost": "${:,.2f}",
    "campaign_cost": "${:,.2f}",
    "expected_net_value": "${:,.2f}",
}))
top_100_targets.style.format({
    "predicted_churn_probability": "{:.1%}",
    "predicted_ltv_if_retained": "${:,.0f}",
    "expected_value_before_cost": "${:,.2f}",
    "expected_offer_cost": "${:,.2f}",
    "campaign_cost": "${:,.2f}",
    "expected_net_value": "${:,.2f}",
})

,customers_targeted,expected_value_before_cost,outreach_cost,expected_offer_cost,campaign_cost,expected_net_value
0,100,"$46,647.51","$2,000.00","$20,000.00","$22,000.00","$24,647.51"


,CustomerID,predicted_churn_probability,predicted_ltv_if_retained,expected_value_before_cost,expected_offer_cost,campaign_cost,expected_net_value
0,0295-PPHDO,96.9%,"$5,962",$577.86,$200.00,$220.00,$357.86
1,5178-LMXOP,97.6%,"$5,795",$565.49,$200.00,$220.00,$345.49
2,1628-BIZYP,94.8%,"$5,754",$545.50,$200.00,$220.00,$325.50
3,2865-TCHJW,92.9%,"$5,808",$539.72,$200.00,$220.00,$319.72
4,1320-HTRDR,90.5%,"$5,948",$538.35,$200.00,$220.00,$318.35
5,3716-BDVDB,92.8%,"$5,795",$537.88,$200.00,$220.00,$317.88
6,7180-PISOG,95.8%,"$5,554",$532.16,$200.00,$220.00,$312.16
7,6651-AZVTJ,86.9%,"$6,088",$528.91,$200.00,$220.00,$308.91
8,8361-LTMKD,89.7%,"$5,839",$523.48,$200.00,$220.00,$303.48
9,5228-EXCET,86.4%,"$5,964",$515.16,$200.00,$220.00,$295.16
